In [1]:
import numpy as np
import pandas as pd
import pickle

# 1. Data
- Load data from pickle file
- split the data into independent and dependent
- split to train and test set

In [2]:
data = pickle.load(open('data_face_features_emotion.pickle',mode='rb'))

In [3]:
X = np.array(data['data']) # indendepent variable
y = np.array(data['label']) # dependent variable

In [4]:
X.shape , y.shape

((8184, 1, 128), (8184,))

In [5]:
X = X.reshape(-1,128)
X.shape

(8184, 128)

In [6]:
# split the data into train and test
from sklearn.model_selection import train_test_split

In [7]:
x_train,x_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=0)

In [8]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((6547, 128), (1637, 128), (6547,), (1637,))

# 2. Train Machine Learning

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

### Logistic Regression

In [11]:
model_logistic = LogisticRegression(max_iter=1000)
model_logistic.fit(x_train,y_train) # training logistic regression

LogisticRegression(max_iter=1000)

In [12]:
def get_report(model, x_train,y_train,x_test,y_test):
    y_pred_train = model.predict(x_train)
    y_pred_test = model.predict(x_test)

    # accuracy score
    acc_train = accuracy_score(y_train,y_pred_train)
    acc_test = accuracy_score(y_test,y_pred_test)

    # f1 score
    f1_score_train = f1_score(y_train,y_pred_train,average='macro')
    f1_score_test = f1_score(y_test,y_pred_test,average='macro')


    print('Accuracy Train = %0.2f'%acc_train)
    print('Accuracy Test = %0.2f'%acc_test)
    print('F1 Score Train = %0.2f'%f1_score_train)
    print('F1 Score Test = %0.2f'%f1_score_test)

In [13]:
get_report(model_logistic,x_train,y_train,x_test,y_test)

Accuracy Train = 0.34
Accuracy Test = 0.31
F1 Score Train = 0.32
F1 Score Test = 0.29


### Support Vector Machines

In [14]:
model_svc = SVC(probability=True)
model_svc.fit(x_train,y_train)

SVC(probability=True)

In [15]:
get_report(model_svc,x_train,y_train,x_test,y_test)

Accuracy Train = 0.49
Accuracy Test = 0.39
F1 Score Train = 0.49
F1 Score Test = 0.38


### Random Forest

In [16]:
model_rf = RandomForestClassifier(n_estimators=10,)
model_rf.fit(x_train,y_train)

RandomForestClassifier(n_estimators=10)

In [17]:
get_report(model_rf,x_train,y_train,x_test,y_test)

Accuracy Train = 0.99
Accuracy Test = 0.51
F1 Score Train = 0.99
F1 Score Test = 0.52


## Voting Classifier

In [20]:
model_voting = VotingClassifier(estimators=[
    ('logistic',LogisticRegression(max_iter=1000)),
    ('svm',SVC(probability=True)),
    ('rf',RandomForestClassifier())
], voting='soft',weights=[2,3,1])

In [21]:
model_voting.fit(x_train,y_train)

VotingClassifier(estimators=[('logistic', LogisticRegression(max_iter=1000)),
                             ('svm', SVC(probability=True)),
                             ('rf', RandomForestClassifier())],
                 voting='soft', weights=[2, 3, 1])

In [22]:
get_report(model_voting,x_train,y_train,x_test,y_test)

Accuracy Train = 0.75
Accuracy Test = 0.45
F1 Score Train = 0.75
F1 Score Test = 0.44


# 3. Parameter Tuning

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:
model_grid = GridSearchCV(model_voting,
                         param_grid={
                             'svm__C':[3,5,7,10],
                             'svm__gamma':[0.1,0.3,0.5],
                             'rf__n_estimators':[5,10,20],
                             'rf__max_depth':[3,5,7],
                             'voting':['soft','hard']
                         },scoring='accuracy',cv=3,n_jobs=1,verbose=2)

In [25]:
model_grid.fit(x_train,y_train)

Fitting 3 folds for each of 216 candidates, totalling 648 fits
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  23.0s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  22.4s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  22.1s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  21.9s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  21.7s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  21.9s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  20.5s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  20.4s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  20.5s
[

GridSearchCV(cv=3,
             estimator=VotingClassifier(estimators=[('logistic',
                                                     LogisticRegression(max_iter=1000)),
                                                    ('svm',
                                                     SVC(probability=True)),
                                                    ('rf',
                                                     RandomForestClassifier())],
                                        voting='soft', weights=[2, 3, 1]),
             n_jobs=1,
             param_grid={'rf__max_depth': [3, 5, 7],
                         'rf__n_estimators': [5, 10, 20],
                         'svm__C': [3, 5, 7, 10], 'svm__gamma': [0.1, 0.3, 0.5],
                         'voting': ['soft', 'hard']},
             scoring='accuracy', verbose=2)

In [26]:
model_best_estimator = model_grid.best_estimator_

In [27]:
model_grid.best_score_

0.38719970261550835

# 4. Save Model

In [28]:
pickle.dump(model_best_estimator,open('./models/machinelearning_face_emotion.pkl',mode='wb'))